In [ ]:
import os
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score

In [ ]:
# Load data
train_df = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
test_df = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')
sample_sub = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')

In [ ]:
# Inputs
#train_df['input'] = train_df['text']
#test_df['input'] = test_df['text']
train_df['input'] = train_df['keyword'].fillna('') + ' ' + train_df['text']
test_df['input'] = test_df['keyword'].fillna('') + ' ' + test_df['text']
train_ds = Dataset.from_pandas(train_df[['input', 'target']].rename(columns={'target': 'labels'}))
test_ds = Dataset.from_pandas(test_df[['input']])

In [ ]:
# Model and tokenizer
#model_name = 'microsoft/deberta-v3-small'
#model_name = 'microsoft/deberta-v3-base'
model_name = 'microsoft/deberta-v3-large'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

In [ ]:
# Tokenize
def tokenize(batch): return tokenizer(batch['input'], truncation=True, padding=True)
train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

In [ ]:
# Train-test split
#split = train_ds.train_test_split(test_size=0.01, seed=42)
split = train_ds.train_test_split(test_size=0.05, seed=42)

In [ ]:
# Training arguments
# args = TrainingArguments(
#     output_dir='outputs',
#     eval_strategy='epoch',
#     per_device_train_batch_size=32,
#     per_device_eval_batch_size=64,
#     num_train_epochs=1,
#     learning_rate=8e-5,
#     weight_decay=0.01,
#     fp16=True,
#     report_to='none'
# )

args = TrainingArguments(
    output_dir='outputs',
    eval_strategy="steps",
    eval_steps=50,  
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=2,            
    learning_rate=3e-5,             
    weight_decay=0.01,
    warmup_ratio=0.1,              
    lr_scheduler_type='cosine',     
    fp16=True,
    report_to='none'
)

In [ ]:
# Metric
def compute_metrics(p): return {'f1': f1_score(p.label_ids, np.argmax(p.predictions, axis=1), zero_division=0)}

In [ ]:
# Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
# Train
trainer.train()

In [ ]:
# Predict
preds = trainer.predict(test_ds).predictions
labels = np.argmax(preds, axis=1)

In [ ]:
# Submit
sample_sub['target'] = labels
sample_sub.to_csv('submission.csv', index=False)

In [ ]:
# Evaluate on the small test split
eval_result = trainer.evaluate()
print(f"Validation F1 Score: {eval_result['eval_f1']:.4f}")